# 07 — Online RL Fine-Tuning of the SFT'd SLM with GRPO · Google Colab

**Phase 3 (RL stage)** · MSc Thesis — ECLIPSE

**Goal.** Take the LoRA-SFT'd Gemma-4 E4B from notebook 05 and improve it with **online environment-reward RL** in CityLearn. The SAC teacher is weak (≈20 episodes), so SFT alone leaves headroom; this notebook closes the gap by sampling rollouts from the current policy, scoring them with the real `MERLINReward` from CityLearn, and updating the LoRA weights with a **GRPO-style group-relative policy gradient**.

**Algorithm — GRPO (group-relative policy optimization) adapted for a multi-step environment.**

- Each "training instance" is a `(t0, building_slice)` pair drawn from a seasonal-stratified pool.
- For each instance we sample **G parallel rollouts** of **W steps** from the same `t0` using the current policy. Same `t0` across the group is what makes the group-mean a valid variance reducer (Shao et al., DeepSeekMath 2024).
- Per rollout we record `(state_text_t, completion_t, log_probs_t, reward_t)` for every step. Return per rollout `R_g = Σ_t reward_t` (window-sum MERLIN, dense, negative cost).
- Group-normalized advantage `A_g = (R_g − mean_R) / (std_R + ε)`.
- Loss `= − E_{g,t} [ A_g · log π_θ(completion_t | state_t) ] + β · KL(π_θ || π_ref)`.
- π_ref is the **frozen SFT'd LoRA** — RL is only allowed to drift so far from the supervised prior (RLHF-standard KL safeguard; see RAGEN/StarPO 2025 for the multi-turn version).

**Why GRPO and not PPO or RLOO.** PPO needs a value head — a second adapter on a 4-bit Gemma is fragile and Unsloth's pipeline isn't built for it. RLOO is algorithmically close but TRL's `RLOOTrainer` is single-prompt-oriented and lags GRPO in env-coupled tooling. GRPO is critic-free, stable at LoRA scale, and the standard for env-reward LLM-agent RL (RAGEN, ArCHer, ReFT all converge here).

**Why a custom training loop, not `trl.GRPOTrainer`.** `GRPOTrainer` treats one completion as one full action. Our env is multi-step (8,760 hours/year); we want the group baseline applied to a **window** of W actions per rollout, with rewards accumulated across env steps. That requires interleaving generation with `env.step`, which the TRL trainer abstraction doesn't expose. A clean custom loop is more transparent for the thesis anyway.

**Key references** — DeepSeekMath/GRPO ([Shao et al. 2024](https://arxiv.org/abs/2402.03300)), RAGEN/StarPO ([Wang et al. 2025](https://arxiv.org/abs/2504.20073)), ArCHer ([Zhou et al. 2024](https://arxiv.org/abs/2402.19446)), ReFT ([Luong et al. 2024](https://arxiv.org/abs/2401.08967)), RLOO ([Ahmadian et al. 2024](https://arxiv.org/abs/2402.14740)).

**Pre-requisite.** Notebook 05 must have produced a LoRA adapter at `MyDrive/eclipse-thesis/sft_adaptersV3/lora_adapter` (or equivalent). If it doesn't exist this notebook falls back to RL-from-base-Gemma — useful for sanity checks but not the thesis condition.

**Compute.** T4 (free Colab Pro): toy defaults below run in ~30–45 min and produce a smoke-tested run. A100 with `FULL_SCALE = True` is the configuration the thesis numbers come from (~6 h).

## § 0 — Runtime & Config

> **Edit this cell only.** Nothing else needs changing.

Defaults are **toy-scale**: small `G`, short `W`, few `UPDATES`. The whole loop runs in under an hour on a T4 so you can see end-to-end behavior before paying for a long run. Flip `FULL_SCALE = True` (or override individual constants below) when you graduate to A100.

In [ ]:
# CRITICAL: import unsloth before transformers / trl / peft so its kernel
# patches actually attach. Same rule as in nb 05.
try:
    import unsloth  # noqa: F401
except ImportError:
    pass  # § 1 install will follow on a fresh kernel

import os, sys, subprocess, time, warnings, json, random, math
from pathlib import Path
import numpy as np

# ── Repo ──────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/antonisbast/eclipse-thesis.git"
REPO_DIR    = "/content/eclipse-thesis"

# ── SFT'd LoRA adapter (produced by nb 05) ────────────────────────────────
# Glob is searched after Drive mount in § 3. Newest match wins.
SFT_ADAPTER_GLOB = "/content/drive/MyDrive/eclipse-thesis/sft_adaptersV3/**/lora_adapter"
ALLOW_RL_FROM_BASE = True   # if no SFT adapter found, RL from the bare base Gemma

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_SEQ_LEN:    int  = 1024
MAX_NEW_TOKENS: int  = 120   # <thought> ~80 + action block ~30 + slack

# ── LoRA — MUST match the SFT'd adapter rank/alpha so it loads cleanly ───
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05   # small dropout helps RL exploration vs the 0.0 used at SFT

# ── GRPO hyperparameters ─────────────────────────────────────────────────
FULL_SCALE = False     # flip True on A100 for thesis-scale run

if FULL_SCALE:
    GROUP_SIZE   = 4    # G — rollouts per update sharing the same t0
    WINDOW_STEPS = 48   # W — one diurnal cycle
    UPDATES      = 300  # total optimizer updates
    EVAL_EVERY   = 25
    CHECKPOINT_EVERY = 25
else:
    GROUP_SIZE       = 2
    WINDOW_STEPS     = 12
    UPDATES          = 8
    EVAL_EVERY       = 4
    CHECKPOINT_EVERY = 4

KL_BETA        = 0.04         # KL(policy ‖ frozen SFT'd ref). Halve if KL > 20 nats; double if reward collapses.
LEARNING_RATE  = 5e-6         # Unsloth Gemma-4 GRPO default. Raise to 1e-5 after 50 stable updates.
GRAD_CLIP      = 1.0          # StarPO-S stabilizer (RAGEN)
ENTROPY_COEF   = 0.0          # off by default; small (1e-3) helps if rollouts collapse to one token
FALLBACK_PENALTY = 0.5        # added to per-step reward when parsing fails (positive value = penalty applied as negative)

# Sampling at rollout time. Greedy at eval time.
ROLLOUT_TEMPERATURE = 0.7
ROLLOUT_TOP_P       = 0.9

# ── Episode geometry ─────────────────────────────────────────────────────
# t0 is sampled from a seasonal-stratified pool so the policy doesn't overfit to summer.
# 8 760 hours / year; one window must fit entirely inside the year.
T0_POOL_SEASONS = {
    "winter": (0,    2160),   # Jan-Mar
    "spring": (2160, 4320),   # Apr-Jun
    "summer": (4320, 6480),   # Jul-Sep
    "autumn": (6480, 8640),   # Oct-Dec — leave headroom for WINDOW_STEPS
}

# ── Evaluation slice (between-update KPI probe; cheap, deterministic) ────
EVAL_START = 4000               # summer week with batteries primed by then
EVAL_LEN   = 168                # 1 week. Set to 8 760 for the final full-year eval.

# ── Building slices (Phase 3 single-agent, group-centralized over 3) ─────
# Imported from src.env in § 6; mirrored here so the config cell is self-contained.
N_BUILDINGS    = 3
TRAIN_SLICE    = [0, 1, 2]      # in-distribution
HELDOUT_SLICE  = [3, 4, 5]      # generalization probe (also Phase 4 agent β)

# ── CityLearn ─────────────────────────────────────────────────────────────
CITYLEARN_VERSION = "2.6.0b2"

# ── Output ────────────────────────────────────────────────────────────────
RUN_NAME    = time.strftime("grpo_%Y%m%d_%H%M%S")
DRIVE_ROOT  = "/content/drive/MyDrive/eclipse-thesis"
OUT_DIR     = f"{DRIVE_ROOT}/rl_runs/{RUN_NAME}"

SEED = 42
HF_TOKEN = ""

try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — Runtime → Change runtime type → GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")

print(f"Scale: {'FULL' if FULL_SCALE else 'TOY'}  |  G={GROUP_SIZE}  W={WINDOW_STEPS}  updates={UPDATES}")

## § 1 — Install Dependencies

Same recipe as nb 05 — CityLearn 2.6.0b2 pinned (for `evaluate_v2`), Unsloth + TRL pinned to the versions that play with Gemma-4 4-bit + LoRA. We don't strictly need TRL's GRPOTrainer here (we write our own loop) but we keep TRL installed for `DataCollatorForCompletionOnlyLM` patterns and for consistency with the rest of the pipeline.

In [ ]:
# CityLearn 2.6 (pre-release). --no-deps because it pulls heavy EnergyPlus extras.
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

In [ ]:
!pip install -q --upgrade pip
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece torchao
!pip install -q "trl>=0.15,<0.20"
!pip install -q triton==3.6.0

import trl, transformers, peft
print(f"✓ trl={trl.__version__}  transformers={transformers.__version__}  peft={peft.__version__}")

## § 1.5 — Optional Performance Deps (best-effort)

In [ ]:
import subprocess, sys, importlib

def _try_install(label, pip_args):
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pip_args,
                           capture_output=True, text=True, timeout=600)
        ok = r.returncode == 0
        print(f"{'✓' if ok else '✗'} {label}")
        return ok
    except Exception as e:
        print(f"✗ {label}  ({type(e).__name__}: {e})")
        return False

_try_install("hf_transfer", ["hf_transfer"]); os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
_try_install("xformers",    ["xformers"])
_try_install("flash-attn (best effort)", ["flash-attn", "--no-build-isolation"])

## § 2 — Clone Repo (pulls `src/`)

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

## § 3 — Mount Drive + Locate the SFT'd LoRA Adapter

The RL phase starts from the SFT'd LoRA produced by nb 05. We also save every checkpoint here so a Colab disconnect doesn't nuke the run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
matches = sorted(glob.glob(SFT_ADAPTER_GLOB, recursive=True))
if matches:
    SFT_ADAPTER_PATH = matches[-1]
    print(f"✓ SFT adapter found: {SFT_ADAPTER_PATH}")
elif ALLOW_RL_FROM_BASE:
    SFT_ADAPTER_PATH = None
    print("⚠  No SFT adapter found — RL will start from the bare base Gemma.")
    print("    Useful for plumbing tests; not the thesis condition.")
else:
    raise FileNotFoundError(f"No SFT adapter under {SFT_ADAPTER_GLOB}. Run nb 05 first.")

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Run dir: {OUT_DIR}")

## § 4 — Load Gemma + Attach LoRA (initialize from SFT'd adapter)

Unsloth's `FastModel.from_pretrained` returns the patched 4-bit base. We then call `get_peft_model` to create the trainable LoRA. If the SFT adapter exists we **overwrite the freshly-initialized LoRA weights with the SFT'd ones** — that way the trainable adapter (the one we'll backprop into) starts from the SFT prior, not random.

We do *not* merge the SFT LoRA into the base; the same set of LoRA weights serves as both the trainable policy adapter and (via a separate state-dict snapshot taken in § 5) the frozen reference policy used in the KL term.

In [ ]:
from unsloth import FastModel
import torch

torch.cuda.empty_cache()

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base loaded in {time.time()-_t0:.1f}s")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = SEED,
)
model.print_trainable_parameters()

# ── Warm-start the trainable LoRA from the SFT adapter ───────────────────
# CAVEAT: `get_peft_model` above has already attached a fresh adapter named
# "default" whose B-matrices are zero (adapter outputs nothing yet). To
# warm-start from SFT we need to OVERWRITE that adapter's weights — NOT
# attach a second adapter and NOT silently fall back to the fresh init.
#
# The earlier approach used `model.load_adapter(path, adapter_name="default",
# is_trainable=True)` inside a try/except. With recent peft this either errors
# (adapter "default" already exists) or silently keeps the fresh init — and
# the except clause swallowed both. We instead stamp the SFT state_dict into
# the existing default adapter via `set_peft_model_state_dict`, then ASSERT
# the weights actually changed before training.
if SFT_ADAPTER_PATH is not None:
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file as load_safetensors

    # Pick any lora_A weight to use as a "did anything change?" probe.
    def _first_lora_A_name(m):
        for k, _ in m.named_parameters():
            if "lora_A" in k:
                return k
        raise RuntimeError("No lora_A parameter — peft adapter did not attach.")

    _ref_name = _first_lora_A_name(model)
    _before   = dict(model.named_parameters())[_ref_name].detach().clone().float().cpu()

    sft_file = (
        os.path.join(SFT_ADAPTER_PATH, "adapter_model.safetensors")
        if os.path.exists(os.path.join(SFT_ADAPTER_PATH, "adapter_model.safetensors"))
        else os.path.join(SFT_ADAPTER_PATH, "adapter_model.bin")
    )
    if sft_file.endswith(".safetensors"):
        sft_sd = load_safetensors(sft_file)
    else:
        sft_sd = torch.load(sft_file, map_location="cpu")

    incompat = set_peft_model_state_dict(model, sft_sd, adapter_name="default")
    n_missing    = len(getattr(incompat, "missing_keys", []) or [])
    n_unexpected = len(getattr(incompat, "unexpected_keys", []) or [])
    print(f"Stamped SFT state_dict → adapter \"default\"  "
          f"(missing={n_missing}, unexpected={n_unexpected})")

    _after = dict(model.named_parameters())[_ref_name].detach().float().cpu()
    delta  = (_after - _before).abs().mean().item()
    print(f"  Δ|{_ref_name.rsplit('.', 2)[-2]}| = {delta:.3e}  (must be > 0)")
    assert delta > 1e-8, (
        "LoRA warm-start did NOT change any weights — the SFT prior was "
        "not loaded. Refusing to continue: this would silently degrade to "
        "RL-from-base. Inspect adapter key naming / state_dict prefix."
    )
    print(f"✓ Warm-started LoRA from {SFT_ADAPTER_PATH}")
else:
    # cell 11 already raised if SFT_ADAPTER_PATH is None and
    # ALLOW_RL_FROM_BASE=False — so reaching here means the user explicitly
    # opted into plumbing-only mode.
    print("⚠  ALLOW_RL_FROM_BASE=True — running with the fresh LoRA. "
          "This is a plumbing test only; not the thesis condition.")


## § 5 — Snapshot the Reference Policy (for KL)

We need π_ref(action | state) at every loss computation. Memory budget on T4 forbids loading a second full 4-bit Gemma. Trick: keep the **LoRA-disabled** forward pass of the same model as the reference. When LoRA adapters are disabled, the model is exactly the base Gemma + the *frozen* SFT state-dict we loaded above — i.e. our reference policy.

Concretely: at training time we run two forward passes per batch on the same model — one with adapters enabled (policy logits, gradient-bearing) and one with adapters disabled via `model.disable_adapter()` context manager (reference logits, `torch.no_grad`). This is the trick Unsloth + TRL use under the hood for KL on LoRA.

**However**, this only gives us π_ref = base Gemma, not π_ref = SFT'd Gemma. To get the latter, we make an immutable snapshot of the SFT'd LoRA weights *now* and keep them as a frozen tensor dict, then load them on-the-fly into the model for the reference forward pass (toggling back to the trainable copy for the policy forward pass).

For simplicity in this notebook we take a pragmatic shortcut: **π_ref = the LoRA-disabled base model**. This is a weaker KL anchor (it pulls the policy back toward base Gemma, not toward SFT Gemma), and is acceptable for a smoke run; the markdown below documents the upgrade path. If the policy collapses (reward variance → 0) or KL explodes, swap to the SFT-snapshot reference.

In [ ]:
# Sanity check: disable_adapter() context works and produces different logits.
from src.env import snapshot_state

_probe = "STATE:\nMonth 7, Wed 14:00  |  price=0.540 (PEAK)  |  carbon=0.180 (MID)\nBuildings:\n  B0: SoC= 50.0%  load=2.50 kWh  last_net=+1.00 kWh  solar=HIGH\n  B1: SoC= 40.0%  load=2.00 kWh  last_net=+0.50 kWh  solar=HIGH\n  B2: SoC= 60.0%  load=3.00 kWh  last_net=+1.50 kWh  solar=MID\n"
_msgs  = [{"role":"user", "content": _probe}]
_enc   = tokenizer.apply_chat_template(_msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to(model.device)

model.eval()
with torch.no_grad():
    logits_pol = model(**_enc).logits[0, -1]
    with model.disable_adapter():
        logits_ref = model(**_enc).logits[0, -1]

delta = (logits_pol - logits_ref).abs().mean().item()
print(f"|logit_policy − logit_ref|_mean = {delta:.4f}  (must be > 0 if LoRA is doing anything)")
assert delta > 1e-6, "LoRA adapter is not active — KL will be a no-op."

## § 6 — Env Factory, Prompt, Action Parser

Same primitives the rest of the pipeline uses. The RL training loop talks to CityLearn through `make_colab_env` (string schema, auto-download in Colab) and to the policy through `render_state` / `parse_actions`. The prompt is the **canonical CoT prompt** from `src.agent.make_minimal_prompt(N_BUILDINGS)` — RL training and inference use the IDENTICAL template, eliminating the SFT↔eval drift documented in nb 05 § 19.

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from src.env import (
    make_env, snapshot_state, MERLINReward, OBSERVATIONS, ACTIVE_ACTIONS,
    TRAINING_BUILDINGS, HELDOUT_BUILDINGS, BUILDINGS,
)
from src.agent import render_state, parse_actions, ACTION_RE
from src.sft   import make_sft_prompt
from src.eval  import evaluate, comparison_table

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

# CRITICAL: use the SAME prompt the SFT adapter was trained on. The earlier
# version used `make_minimal_prompt` (CoT, with a `<thought>` block), but
# the SFT adapter has never been taught to emit `<thought>` — that prompt
# would produce either garbage thought tokens (blowing the budget) or
# actions stuck inside the thought block (invisible to parse_actions). The
# OOD prompt was the cause of the "CoT eval blowup" mentioned in nb 05 § 19.
SYSTEM_PROMPT = make_sft_prompt(N_BUILDINGS)
print(f"System prompt: {len(SYSTEM_PROMPT.split())} words  (no-CoT — matches SFT).\n")


def make_colab_env(start: int, end: int, buildings = None) -> CityLearnEnv:
    """Use the project-wide src.env.make_env so the schema (and its local
    root_directory) match nb 04/05/06. The repo was cloned in § 2 so the
    `data/citylearn_datasets/...` tree is on disk."""
    return make_env(
        buildings = buildings if buildings is not None else TRAINING_BUILDINGS,
        start     = start,
        end       = end,
        reward_fn = "merlin",
    )

# Quick smoke: build a 1-step env, render a state, see the format.
_env = make_colab_env(start=4000, end=4001, buildings=TRAIN_SLICE)
_env.reset()
print("--- example rendered state ---")
print(render_state(snapshot_state(_env))[:400])


## § 7 — Rollout Primitive

`rollout_window(t0, slice, sample=True)` runs the **current policy** for `WINDOW_STEPS` env steps starting at `t0` on the given building slice and returns a structured record:

```
{
  "prompts":      [str] × W,        # rendered STATE block at each step
  "input_ids":    [LongTensor] × W, # tokenized prompt with chat template
  "gen_ids":      [LongTensor] × W, # generated completion tokens (variable length)
  "rewards":      [float] × W,      # per-step district-sum MERLIN (incl. fallback penalty)
  "fallbacks":    [bool] × W,       # parse failed → IDLE
  "actions":      [list[float]] × W,
  "return":       float,            # Σ rewards
}
```

Generation is **stochastic** (`temperature`, `top_p`) so the G rollouts in a group actually differ. We use Unsloth's `for_inference` patched generation path; switching back to training mode is a single line.

In [ ]:
def _build_prompt_ids(state_text: str) -> torch.LongTensor:
    """Build chat-templated prompt input_ids for the current state."""
    # Gemma rejects role='system' — fold SYSTEM_PROMPT into the user turn.
    msgs = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nSTATE:\n{state_text}"}]
    enc = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    return enc["input_ids"].to(model.device)


def _generate(prompt_ids: torch.LongTensor, sample: bool) -> torch.LongTensor:
    """Generate a completion. Returns only the NEW token ids (no prompt)."""
    with torch.no_grad():
        out = model.generate(
            input_ids=prompt_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=sample,
            temperature=ROLLOUT_TEMPERATURE if sample else 1.0,
            top_p=ROLLOUT_TOP_P if sample else 1.0,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return out[0, prompt_ids.shape[1]:].detach()


def rollout_window(t0: int, slice_ids, sample: bool = True) -> dict:
    """One rollout of WINDOW_STEPS steps starting at t0 on the given building slice."""
    # CityLearn's simulation_end_time_step is INCLUSIVE — end = t0 + W - 1
    # gives exactly WINDOW_STEPS available env.step() calls. The earlier
    # code used `end = t0 + WINDOW_STEPS`, which provisioned W+1 steps but
    # ran only W — one wasted env step per rollout and a sneaky off-by-one
    # when the loop hit the inclusive boundary.
    env = make_colab_env(start=t0, end=t0 + WINDOW_STEPS - 1, buildings=slice_ids)
    env.reset()

    rec = {"prompts": [], "input_ids": [], "gen_ids": [],
           "rewards": [], "fallbacks": [], "actions": []}
    n_b = len(env.buildings)

    FastModel.for_inference(model)
    for step in range(WINDOW_STEPS):
        snap   = snapshot_state(env)
        s_text = render_state(snap)
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=sample)
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)

        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b); fb = False
        else:
            acts = [0.0] * n_b;             fb = True

        _, r_list, term, trunc, _ = env.step([[float(a)] for a in acts])
        step_r = float(sum(r_list)) - (FALLBACK_PENALTY if fb else 0.0)

        rec["prompts"  ].append(s_text)
        rec["input_ids"].append(p_ids.squeeze(0).cpu())
        rec["gen_ids"  ].append(g_ids.cpu())
        rec["rewards"  ].append(step_r)
        rec["fallbacks"].append(fb)
        rec["actions"  ].append(acts)
        if term or trunc:
            break

    rec["return"] = float(sum(rec["rewards"]))
    return rec

# Smoke-test one rollout (toy: WINDOW_STEPS=12, so ~60-90 s on T4)
print("Smoke-testing one rollout …")
_t0 = time.time()
_r  = rollout_window(t0=4500, slice_ids=TRAIN_SLICE, sample=True)
print(f"  {WINDOW_STEPS} steps in {time.time()-_t0:.1f}s  |  return={_r['return']:.3f}  |  fallbacks={sum(_r['fallbacks'])}")


## § 8 — GRPO Loss and One Optimizer Step

Given a group of G rollouts that all started at the same `t0`, the GRPO loss for the policy is:

$$
\mathcal{L} \;=\; -\,\mathbb{E}_{g,t}\!\left[ A_g \cdot \log \pi_\theta(c_{g,t}\mid s_{g,t}) \right] \;+\; \beta\,\mathrm{KL}\!\left(\pi_\theta \,\Vert\, \pi_\mathrm{ref}\right)
$$

with $A_g = (R_g - \bar R)/(\sigma_R + \epsilon)$, $R_g = \sum_t r_{g,t}$. Concretely, for each `(prompt_ids, gen_ids)` pair we:

1. Re-feed `[prompt_ids, gen_ids]` through the **policy** (LoRA on) to get per-token log-probabilities under the current policy.
2. Re-feed the same sequence through the **reference** (LoRA off via `disable_adapter()`, no grad) for π_ref log-probs.
3. Sum log-probs over the completion tokens only → one log π(c|s) scalar per (g, t).
4. KL term: `(log π − log π_ref).mean()` over completion tokens, micro-batch averaged.
5. Multiply each step's log-prob by the group advantage `A_g` (broadcast within rollout), sum, negate, add KL.

All completion tokens — including `<thought>` reasoning — receive the same advantage. This is intentional: we want RL signal to shape the rationale too (RAGEN, DeepSeek-R1, ReFT). The KL term prevents the rationale from collapsing into degenerate filler.

In [ ]:
import torch.nn.functional as F

def _logprobs_of_completion(prompt_ids: torch.Tensor,
                            gen_ids: torch.Tensor,
                            with_grad: bool) -> torch.Tensor:
    """Return per-completion-token log-probabilities under the CURRENT model state.
    Caller controls whether LoRA adapters are enabled via with model.disable_adapter().
    """
    full = torch.cat([prompt_ids, gen_ids], dim=0).unsqueeze(0).to(model.device)
    ctx  = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        logits = model(full).logits[0]   # [T, V]
    # Shift so logits at position i predict token at i+1
    p_len = prompt_ids.shape[0]
    g_len = gen_ids.shape[0]
    # Logits aligned with completion tokens are at positions [p_len-1 .. p_len+g_len-2]
    comp_logits = logits[p_len-1 : p_len-1 + g_len]                  # [g_len, V]
    log_probs   = F.log_softmax(comp_logits.float(), dim=-1)
    targets     = gen_ids.to(model.device).unsqueeze(-1)              # [g_len, 1]
    return log_probs.gather(-1, targets).squeeze(-1)                  # [g_len]


def grpo_loss(group_records: list[dict]):
    """Compute the GRPO loss for one group of G rollouts sharing a t0."""
    returns = torch.tensor([r["return"] for r in group_records], dtype=torch.float32)
    adv     = (returns - returns.mean()) / (returns.std() + 1e-6)     # [G]

    pg_terms, kl_terms = [], []
    FastModel.for_training(model)

    for g, rec in enumerate(group_records):
        for p_ids, g_ids in zip(rec["input_ids"], rec["gen_ids"]):
            if g_ids.numel() == 0:
                continue
            lp_pol = _logprobs_of_completion(p_ids, g_ids, with_grad=True)
            with model.disable_adapter():
                lp_ref = _logprobs_of_completion(p_ids, g_ids, with_grad=False)

            # Policy gradient term: − A_g · Σ_t log π(c_t|s_t)
            pg_terms.append(-adv[g] * lp_pol.sum())

            # KL term: k3 estimator (Schulman, http://joschu.net/blog/kl-approx.html).
            # The naive `(lp_pol - lp_ref).mean()` is unbiased only in expectation
            # under samples from π_pol — for any finite sample it can go NEGATIVE,
            # in which case the optimiser is REWARDED for moving away from the
            # reference. The k3 estimator is element-wise ≥ 0 and is what TRL /
            # DeepSpeed use in production GRPO/PPO loops:
            #     KL ≈ exp(log_ratio) − 1 − log_ratio
            log_ratio  = lp_pol - lp_ref
            # Clamp to avoid `exp` overflow when the policy has drifted far from
            # the reference (a few tokens with log_ratio ≫ 0 would otherwise
            # blow up the loss). 10 nats per token is a generous ceiling.
            log_ratio  = log_ratio.clamp(min=-10.0, max=10.0)
            kl_per_tok = log_ratio.exp() - 1.0 - log_ratio     # ≥ 0 element-wise
            kl_terms.append(kl_per_tok.mean())

    if not pg_terms:
        return None, {"pg": 0.0, "kl": 0.0, "loss": 0.0}

    pg = torch.stack(pg_terms).mean()
    kl = torch.stack(kl_terms).mean()
    loss = pg + KL_BETA * kl
    return loss, {"pg": pg.item(), "kl": kl.item(), "loss": loss.item()}


## § 9 — Training Loop

Seasonal-stratified `t0` sampling, G parallel rollouts per update, GRPO step, log to a JSONL on Drive every update, full-year-week eval every `EVAL_EVERY` updates, adapter checkpoint to Drive every `CHECKPOINT_EVERY`.

In [ ]:
from torch.optim import AdamW

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable, lr=LEARNING_RATE)

rng = random.Random(SEED)
_season_keys = list(T0_POOL_SEASONS.keys())

def sample_t0() -> int:
    season = rng.choice(_season_keys)
    lo, hi = T0_POOL_SEASONS[season]
    hi = min(hi, 8759 - WINDOW_STEPS - 1)
    return rng.randint(lo, hi)


def quick_eval(label: str) -> dict:
    """Deterministic 1-week eval on TRAIN_SLICE — cheap probe between updates."""
    env = make_colab_env(start=EVAL_START, end=EVAL_START + EVAL_LEN - 1,
                         buildings=TRAIN_SLICE)
    env.reset()
    n_b, n_fb, total_r = len(env.buildings), 0, 0.0
    done = False
    while not done:
        snap   = snapshot_state(env)
        s_text = render_state(snap)
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=False)        # greedy
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)
        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b)
        else:
            acts = [0.0]*n_b; n_fb += 1
        _, r_list, term, trunc, _ = env.step([[float(a)] for a in acts])
        total_r += float(sum(r_list))
        done = bool(term or trunc)
    res = evaluate(env, label=label)
    return {"label": label, "phase1": res.phase1, "combined": res.combined,
            "reward_sum": total_r, "fallbacks": n_fb}


log_path = f"{OUT_DIR}/train_log.jsonl"
Path(log_path).parent.mkdir(parents=True, exist_ok=True)
print(f"Logging to {log_path}")
print(f"Starting GRPO training — {UPDATES} updates, G={GROUP_SIZE}, W={WINDOW_STEPS}\n")

history = []
for update in range(1, UPDATES + 1):
    upd_t0 = time.time()
    t0     = sample_t0()

    # ── 1. Collect G rollouts from the same t0 ───────────────────────────
    group = []
    for g in range(GROUP_SIZE):
        group.append(rollout_window(t0=t0, slice_ids=TRAIN_SLICE, sample=True))

    returns = [r["return"] for r in group]
    fb_rate = sum(sum(r["fallbacks"]) for r in group) / max(1, GROUP_SIZE * WINDOW_STEPS)

    # ── 2. GRPO loss + step ──────────────────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    loss, info = grpo_loss(group)
    if loss is not None:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        optimizer.step()

    # ── 3. Log ───────────────────────────────────────────────────────────
    rec = {
        "update":      update,
        "t0":          t0,
        "returns":     returns,
        "return_mean": float(np.mean(returns)),
        "return_std":  float(np.std(returns)),
        "fallback_rate": fb_rate,
        "wall_s":      time.time() - upd_t0,
        **info,
    }
    history.append(rec)
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")

    print(f"upd {update:3d}/{UPDATES}  t0={t0:5d}  R̄={rec['return_mean']:+.3f}±{rec['return_std']:.3f}  "
          f"loss={info['loss']:+.4f}  KL={info['kl']:+.3f}  fb={fb_rate*100:.1f}%  ({rec['wall_s']:.0f}s)")

    # ── 4. Eval probe ────────────────────────────────────────────────────
    if update % EVAL_EVERY == 0 or update == UPDATES:
        ev = quick_eval(label=f"rl_upd{update}")
        print(f"  ↳ eval week (greedy): Phase I={ev['phase1']:.4f}  Combined={ev['combined']:.4f}  fb={ev['fallbacks']}/{EVAL_LEN}")
        with open(log_path, "a") as f:
            f.write(json.dumps({"eval": ev, "update": update}) + "\n")

    # ── 5. Checkpoint to Drive ───────────────────────────────────────────
    if update % CHECKPOINT_EVERY == 0 or update == UPDATES:
        ck = f"{OUT_DIR}/lora_adapter_upd{update}"
        model.save_pretrained(ck)
        tokenizer.save_pretrained(ck)
        print(f"  ↳ checkpoint → {ck}")

print("\nTraining done.")

## § 10 — Final Evaluation: RL vs SFT vs RBC vs No-Control

Run the **full-year** KPI evaluation on (a) the training slice and (b) the held-out slice. This is the canonical thesis number. Toy mode: defaults to `EVAL_LEN_FINAL = 168` (one week) so the cell finishes in minutes; flip to `8760` for the real run.

In [ ]:
from citylearn.agents.rbc import BasicBatteryRBC

EVAL_LEN_FINAL = 168 if not FULL_SCALE else 8758

def run_policy(label, slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length - 1, buildings=slice_ids)
    env.reset()
    n_b, n_fb = len(env.buildings), 0
    done, t = False, 0
    t0 = time.time()
    while not done:
        s_text = render_state(snapshot_state(env))
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=False)
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)
        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b)
        else:
            acts = [0.0]*n_b; n_fb += 1
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc); t += 1
    print(f"[{label}] {t} steps  {time.time()-t0:.0f}s  fallbacks={n_fb}/{t}")
    return env

def run_rbc(slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length-1, buildings=slice_ids)
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env

def run_noop(slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length-1, buildings=slice_ids)
    env.reset(); n_b = len(env.buildings)
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

print("── Evaluating on TRAIN_SLICE [0,1,2] ──")
env_noop  = run_noop(TRAIN_SLICE)
env_rbc   = run_rbc (TRAIN_SLICE)
env_rl    = run_policy("rl-train", TRAIN_SLICE)

print("\n── Evaluating on HELDOUT_SLICE [3,4,5] (generalization) ──")
env_rl_h  = run_policy("rl-heldout", HELDOUT_SLICE)
env_rbc_h = run_rbc(HELDOUT_SLICE)

results = [
    evaluate(env_noop, "No-Control [train]"),
    evaluate(env_rbc,  "RBC [train]"),
    evaluate(env_rl,   "GRPO-SLM [train]"),
    evaluate(env_rbc_h,"RBC [heldout]"),
    evaluate(env_rl_h, "GRPO-SLM [heldout]"),
]
challenge_df, zne_df = comparison_table(results)
print("\nChallenge scores — lower is better; No-Control is the reference.")
display(challenge_df.round(4))
print("\nZNE + self-consumption:")
display(zne_df[["solar generation (kWh)", "grid import (kWh)",
                "ZNE ratio (solar / import)", "self-consumption ratio"]].round(4))

## § 11 — Training Curve

Sanity-check that the policy is actually learning: per-update mean return should trend up (less negative); KL should grow but plateau; fallback rate should fall.

In [ ]:
import matplotlib.pyplot as plt

log_rows = [json.loads(l) for l in open(log_path)]
train_rows = [r for r in log_rows if "update" in r and "return_mean" in r]
eval_rows  = [r["eval"] for r in log_rows if "eval" in r]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = [r["update"] for r in train_rows]
axes[0].plot(x, [r["return_mean"] for r in train_rows], label="mean return (window)")
axes[0].fill_between(x,
    [r["return_mean"] - r["return_std"] for r in train_rows],
    [r["return_mean"] + r["return_std"] for r in train_rows], alpha=0.2)
axes[0].set_xlabel("update"); axes[0].set_ylabel("window return"); axes[0].set_title("Policy return per update")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(x, [r["kl"] for r in train_rows], color="tab:orange")
axes[1].set_xlabel("update"); axes[1].set_ylabel("KL(π‖π_ref) [nats / completion token]")
axes[1].set_title("KL to reference"); axes[1].grid(alpha=0.3)

axes[2].plot(x, [100*r["fallback_rate"] for r in train_rows], color="tab:red")
axes[2].set_xlabel("update"); axes[2].set_ylabel("% steps with parse failure")
axes[2].set_title("Action-parse failure rate"); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

if eval_rows:
    print("\nEval-week KPI progression (greedy, TRAIN_SLICE):")
    import pandas as pd
    display(pd.DataFrame(eval_rows).set_index("label").round(4))

## § 12 — Persist Final Adapter to Drive (canonical name)

In [ ]:
FINAL_DIR = f"{DRIVE_ROOT}/rl_adapters/{RUN_NAME}"
os.makedirs(FINAL_DIR, exist_ok=True)
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Final adapter saved: {FINAL_DIR}")
print("  This is the artifact to load for Phase 4 dual-agent deployment.")

## § 13 — Go / No-Go Checklist before Phase 4

Before promoting this LoRA into the two Phase 4 agents (α on B0–2, β on B3–5), confirm all of the following on the **full-year** rerun of § 10 (set `FULL_SCALE = True`, `EVAL_LEN_FINAL = 8758`, rerun §§ 10–11):

| # | Criterion | Threshold | Where to check |
|---|---|---|---|
| 1 | Phase I KPI vs SAC on training buildings | ≤ 1.3 × SAC (≥ 70 % of SAC perf — CLAUDE.md gate) | § 10 challenge table |
| 2 | GRPO-SLM beats RBC and No-Control on Phase I | strict on both | § 10 |
| 3 | Generalization gap (train vs heldout) | ≤ 20 % relative degradation | § 10 |
| 4 | Action-parse fallback rate, full-year greedy | ≤ 2 % | § 10 print |
| 5 | KL(π ‖ π_ref) at end of training | ≤ 20 nats/seq, plateaued (no monotonic blow-up) | § 11 middle plot |
| 6 | Reward variance non-zero through training (no "Echo Trap" — RAGEN) | std stays > 0.1 × \|mean\| through last 25 % of training | § 11 left plot |
| 7 | Manual rationale sanity (20 sampled `<thought>` blocks) | references plausible signals (load / solar / SoC / price) | nb 06 § 17-style inspection |
| 8 | Checkpoint reproducibility | reloading from `FINAL_DIR` reproduces the § 10 KPI within 1 % | smoke rerun |

Fail any of #1–#3 → iterate before Phase 4. Tuning levers in priority order:
1. **Window length `W`** — too short → policy can't see consequences; too long → return variance explodes.
2. **`KL_BETA`** — halve if KL is exploding; double if reward collapses to single mode.
3. **`LEARNING_RATE`** — drop 2× if loss is noisy.
4. **`FALLBACK_PENALTY`** — raise if parse failures persist past update ~50.
5. **CoT warm-restart** — if the SFT adapter has zero `<thought>` mass, do a brief CoT-SFT pass with synthesized rationales before RL (see Design Rationale in the title cell).